In [18]:
import pandas as pd
import pyodbc
from datetime import datetime
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore') # Ignora warnings, use com cautela em produção

excel_path = 'Tropeira/Marketing/CNPJs Julho - Mateus.xlsx'

# 2. Conexão com SQL Server
server = '######'
database = '####'
username = '####'
password = '####'

conn_str = f'DRIVER={{ODBC Driver 17 for SQL Server}};SERVER={server};DATABASE={database};UID={username};PWD={password}'

try:
    df_cnpjs = pd.read_excel(excel_path, usecols=['CNPJ'])
except FileNotFoundError:
    print(f"Erro: O arquivo Excel não foi encontrado no caminho: {excel_path}")
    exit()
except Exception as e:
    print(f"Erro ao carregar o arquivo Excel: {e}")
    exit()

all_results = []
problematic_cnpjs = [] # Lista para armazenar CNPJs que causam problemas

with pyodbc.connect(conn_str) as conn:
    cursor = conn.cursor()

    for _, row in tqdm(df_cnpjs.iterrows(), total=len(df_cnpjs), desc="Processando CNPJs"):
        cnpj_para_buscar = str(row['CNPJ']).replace('.', '').replace('/', '').replace('-', '')

        sql = """
        WITH CTE AS (SELECT
        A1_CGC,
        D2_CLIENTE,
        FORMAT(CAST(A1_PRICOM as datetime), 'yyyy/MM/dd') A1_PRICOM,
        FORMAT(CAST(D2_EMISSAO as datetime), 'yyyy/MM/dd') D2_EMISSAO,
        A1_NOME,
        DATEDIFF(DAY,
        			CAST(LEAD(D2_EMISSAO, 1) OVER (PARTITION BY D2_CLIENTE ORDER BY CAST(D2_EMISSAO AS DATE) DESC) AS DATE),
        			CAST(D2_EMISSAO AS DATE)) AS COMPRAS_DIFERENCA_DIA
        FROM SD2010
        INNER JOIN SA1010 SA1 ON A1_COD = D2_CLIENTE AND A1_LOJA = D2_LOJA AND A1_FILIAL = '01' 
        WHERE A1_CGC = ? AND YEAR(D2_EMISSAO) = YEAR(GETDATE())),
        
        
        CTE2 AS (SELECT A1_CGC, D2_CLIENTE, A1_PRICOM, D2_EMISSAO, A1_NOME, COMPRAS_DIFERENCA_DIA
        FROM CTE
        WHERE COMPRAS_DIFERENCA_DIA >= 90 OR (MONTH(A1_PRICOM) >=   MONTH(GETDATE()) -1) AND YEAR(A1_PRICOM) = YEAR(GETDATE()))
        
        SELECT * 
        FROM CTE2
        ORDER BY D2_EMISSAO DESC --ORDENANDO PARA PEGAR O ÚLTIMO REGISTRO/ÚLTIMA COMPRA DO CLIENTE
        """
        try:
            cursor.execute(sql, cnpj_para_buscar)
            rows = cursor.fetchall()

            processed_rows = []
            for r in rows:
                # ESSA É A MUDANÇA PRINCIPAL: CONVERTER pyodbc.Row para tuple
                if isinstance(r, pyodbc.Row):
                    r_as_tuple = tuple(r) # Converte o objeto Row em uma tupla nativa
                elif isinstance(r, tuple):
                    r_as_tuple = r # Já é uma tupla, usa como está
                else:
                    # Caso receba algo totalmente inesperado que não seja Row nem tuple
                    print(f"\n--- ATENÇÃO: CNPJ {cnpj_para_buscar} retornou tipo de dado inesperado: {type(r)} ---")
                    print(f"Valor: {r}")
                    problematic_cnpjs.append(f"{cnpj_para_buscar} (tipo inesperado: {type(r)})")
                    continue # Pula esta linha para evitar o erro no pandas

                # Agora que garantimos que é uma tupla (ou tentamos), verificamos o tamanho
                if len(r_as_tuple) == 6:
                    processed_rows.append(r_as_tuple)
                else:
                    print(f"\n--- ATENÇÃO: CNPJ {cnpj_para_buscar} retornou tupla com {len(r_as_tuple)} elementos (esperado 6). ---")
                    print(f"Linha: {r_as_tuple}")
                    problematic_cnpjs.append(f"{cnpj_para_buscar} (número de elementos diferente: {len(r_as_tuple)})")

            all_results.extend(processed_rows)

        except pyodbc.ProgrammingError as e:
            print(f"Erro de SQL (ProgrammingError) para CNPJ {cnpj_para_buscar}: {e}")
            problematic_cnpjs.append(cnpj_para_buscar)
        except Exception as e:
            print(f"Um erro inesperado ocorreu para CNPJ {cnpj_para_buscar}: {e}")
            problematic_cnpjs.append(cnpj_para_buscar)

# ---
# Parte para criar o DataFrame pandas
# ---

columns = ['A1_CGC', 'D2_CLIENTE', 'A1_PRICOM', 'D2_EMISSAO', 'A1_NOME', 'COMPRAS_DIFERENCA_DIA']

print(f"\nNúmero total de linhas coletadas de todos os CNPJs: {len(all_results)}")
print(f"CNPJs com formato de resultado inesperado ou erros: {len(problematic_cnpjs)} encontrados.")
if problematic_cnpjs:
    print(f"Primeiros 10 CNPJs problemáticos: {problematic_cnpjs[:10]}...")

if not all_results:
    print("Atenção: Nenhuma linha válida foi retornada do banco de dados para nenhum dos CNPJs na sua lista.")
    print("O DataFrame resultante estará vazio.")
    df_results = pd.DataFrame(columns=columns)
else:
    df_results = pd.DataFrame(all_results, columns=columns)

print("\nPrimeiras linhas do DataFrame resultante:")
print(df_results.head())

print("\nInformações do DataFrame:")
print(df_results.info())

# Opcional: Salve o DataFrame
# df_results.to_excel('Resultado_CNPJs.xlsx', index=False)
# df_results.to_csv('Resultado_CNPJs.csv', index=False)

Processando CNPJs: 100%|███████████████████████████████████████████████████████████████| 95/95 [00:22<00:00,  4.26it/s]


Número total de linhas coletadas de todos os CNPJs: 33
CNPJs com formato de resultado inesperado ou erros: 0 encontrados.

Primeiras linhas do DataFrame resultante:
           A1_CGC D2_CLIENTE   A1_PRICOM  D2_EMISSAO  \
0  59906288000100     230216  2025/06/06  2025/06/06   
1  53162253000128     230225  2025/06/06  2025/06/06   
2  53162253000128     230225  2025/06/06  2025/06/06   
3  53162253000128     230225  2025/06/06  2025/06/06   
4  59963569000103     230209  2025/06/06  2025/06/06   

                                             A1_NOME  COMPRAS_DIFERENCA_DIA  
0  59.906.288 CARLOS ROBERTO FERREIRA HARTL      ...                    NaN  
1  CASA DE CARNES RL AMORIM LTDA                 ...                    0.0  
2  CASA DE CARNES RL AMORIM LTDA                 ...                    0.0  
3  CASA DE CARNES RL AMORIM LTDA                 ...                    NaN  
4  QUINTAS BISTRO E CERVEJARIA LTDA              ...                    0.0  

Informações do DataFrame:
<c

In [14]:
df_results.head()

,A1_CGC,D2_CLIENTE,A1_PRICOM,D2_EMISSAO,A1_NOME,COMPRAS_DIFERENCA_DIA
0,59906288000100,230216,2025/06/06,2025/06/06,59.906.288 CARLOS ROBERTO FERREIRA HARTL ...,NaN
1,53162253000128,230225,2025/06/06,2025/06/06,CASA DE CARNES RL AMORIM LTDA ...,0.0
2,53162253000128,230225,2025/06/06,2025/06/06,CASA DE CARNES RL AMORIM LTDA ...,0.0
3,53162253000128,230225,2025/06/06,2025/06/06,CASA DE CARNES RL AMORIM LTDA ...,NaN
4,59963569000103,230209,2025/06/06,2025/06/06,QUINTAS BISTRO E CERVEJARIA LTDA ...,0.0


In [25]:
df_results.to_excel('MKTAnalitico.xlsx', index = False) ##Consultar para ver se não há incosistências nos cliente, tem que ter 90 dias 
#+ ou ter primeira compra nos meses recentes

In [24]:
df_clientes = df_results.drop_duplicates(subset=['A1_CGC']).copy()
df_clientes.to_excel('MKT.xlsx', index = False)

In [27]:
#Atualizando registros
for _,row in df_clientes.iterrows():
    
    cnpj_para_buscar1 = str(row['A1_CGC']).replace('.', '').replace('/', '').replace('-', '')
    sql = """
    UPDATE SA1010
    SET A1_SATIV8 = '000907'
    WHERE A1_CGC = ?
    """

    cursor.execute(sql, cnpj_para_buscar1)

conn.commit() 
    
    
    

In [ ]:
#TESTE